#### **Testing and finding synthetic values**

In [13]:
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

What this code is doing (The Logic)
Iterating through "Ghosts": It takes a subset of the synthetic rows (the first 20,000).

Finding the "Donors": For a single synthetic row, it looks at the df_test_real dataframe to see which real rows share the exact same value for any given feature.

The "Unique Match" Filter: It looks for features where the synthetic value matches exactly one real row. This is the "smoking gun"—if only one real row has that specific decimal value, that real row must be the "generator" (the donor) for that specific feature in the synthetic row.

The Result: It creates a list of sets, where each set contains the indices of the real samples that "donated" their DNA to create that specific synthetic sample.


To identify real versus synthetic records in the Santander dataset, you must follow two distinct logic steps: **identification** by global uniqueness and **verification** by cross-referencing values.

### Step 1: Global Identification (Is it Real?)

The goal is to find rows that contain at least one "unique signature" across the entire test set.

| Row | Var_0 | Var_1 | Var_2 | Status | Logic |
| --- | --- | --- | --- | --- | --- |
| **0** | **1.11** | 5.55 | 9.99 | **Real** | **1.11** appears nowhere else in the column. |
| **1** | 2.22 | 6.66 | 9.99 | **Synthetic** | All values (2.22, 6.66, 9.99) appear in other rows. |
| **2** | 2.22 | **7.77** | 9.99 | **Real** | **7.77** is a unique value found only in this row. |

* **Logic:** If a value appears exactly **once** in the 200,000 test rows, the row it belongs to is flagged as **Real**.
* **Result:** Row 0 and Row 2 are saved into `real_samples_indexes`.

---

### Step 2: Value Verification (Who Donated the Data?)

Once you have the list of **Real** rows, you check the **Synthetic** rows to see which real row "donated" its specific feature values.

| Synthetic Row | Feature | Value | Real Donor | Verification |
| --- | --- | --- | --- | --- |
| Row 1 | Var_0 | 1.11 | **Row 0** | Row 0 is the only *Real* row with this value. |
| Row 1 | Var_2 | 9.99 | **Multiple** | Both Row 0 and Row 2 have this; donor is unclear. |

* **Logic:** For a synthetic sample, we look for features where the value matches **exactly one** record in our "Real" subset.
* **Result:** This proves that the synthetic data is just a "Frankenstein" mix created by shuffling the values of the real rows.

In [14]:
test_path = './data/test.csv'

df_test = pd.read_csv(test_path)
df_test.drop(['ID_code'], axis=1, inplace=True)
df_test = df_test.values

unique_samples = []
unique_count = np.zeros_like(df_test)
for feature in tqdm(range(df_test.shape[1])):
    _, index_, count_ = np.unique(df_test[:, feature], return_counts=True, return_index=True)
    unique_count[index_[count_ == 1], feature] += 1

# Samples which have unique values are real the others are fake
real_samples_indexes = np.argwhere(np.sum(unique_count, axis=1) > 0)[:, 0]
synthetic_samples_indexes = np.argwhere(np.sum(unique_count, axis=1) == 0)[:, 0]

print(len(real_samples_indexes))
print(len(synthetic_samples_indexes))



  0%|          | 0/200 [00:00<?, ?it/s]

100000
100000


In [15]:
df_test_real = df_test[real_samples_indexes].copy()

generator_for_each_synthetic_sample = []
# Using 20,000 samples should be enough. 
# You can use all of the 100,000 and get the same results (but 5 times slower)
for cur_sample_index in tqdm(synthetic_samples_indexes[:100000]):
    cur_synthetic_sample = df_test[cur_sample_index]
    potential_generators = df_test_real == cur_synthetic_sample

    # A verified generator for a synthetic sample is achieved
    # only if the value of a feature appears only once in the
    # entire real samples set
    features_mask = np.sum(potential_generators, axis=0) == 1
    verified_generators_mask = np.any(potential_generators[:, features_mask], axis=1)
    verified_generators_for_sample = real_samples_indexes[np.argwhere(verified_generators_mask)[:, 0]]
    generator_for_each_synthetic_sample.append(set(verified_generators_for_sample))

  0%|          | 0/100000 [00:00<?, ?it/s]

In [16]:
# Create a DataFrame with only the real samples and save it to a CSV
df_test_real_final = pd.DataFrame(df_test[real_samples_indexes])
df_test_real_final.to_csv('./data/test_real_only.csv', index=False)

print(f"Successfully saved {len(df_test_real_final)} real samples to 'test_real_only.csv'")

Successfully saved 100000 real samples to 'test_real_only.csv'
